In [2]:
import os


RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["OMP_NUM_THREADS"] = "1"

import warnings
warnings.filterwarnings("ignore")

import json
import random
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


# Load raw data
TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

test_passenger_id = test_raw["PassengerId"].copy()

train_raw["is_train"] = 1
test_raw["is_train"] = 0
test_raw["Transported"] = np.nan

data = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)


# PassengerId features
data["GroupId"] = data["PassengerId"].astype(str).str.split("_").str[0].astype(int)
data["PassengerNo"] = data["PassengerId"].astype(str).str.split("_").str[1].astype(int)

data["GroupSize"] = data.groupby("GroupId")["PassengerId"].transform("count")
data["IsAlone"] = (data["GroupSize"] == 1).astype(int)


# Cabin features
data["Cabin"] = data["Cabin"].fillna("Unknown/Unknown/Unknown")

data["Deck"] = data["Cabin"].astype(str).str.split("/").str[0]
data["CabinNum"] = data["Cabin"].astype(str).str.split("/").str[1]
data["Side"] = data["Cabin"].astype(str).str.split("/").str[2]

data["CabinNum"] = pd.to_numeric(data["CabinNum"], errors="coerce")


# Family name features
data["Name"] = data["Name"].fillna("Unknown Unknown")
data["Surname"] = data["Name"].astype(str).str.split().str[-1]

data["SurnameSize"] = data.groupby("Surname")["PassengerId"].transform("count")
data["HasFamilyName"] = (data["Surname"] != "Unknown").astype(int)


# Missing values and spending features
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

for col in spend_cols:
    data[col] = data[col].fillna(0)

data["TotalSpend"] = data[spend_cols].sum(axis=1)

data.loc[data["CryoSleep"].isna() & (data["TotalSpend"] == 0), "CryoSleep"] = True
data.loc[data["CryoSleep"].isna() & (data["TotalSpend"] > 0), "CryoSleep"] = False

data["VIP"] = data["VIP"].fillna(False)

for col in ["HomePlanet", "Destination"]:
    data[col] = data[col].fillna("Unknown")

data["Age"] = data.groupby(["HomePlanet", "Destination"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

data["Age"] = data["Age"].fillna(data["Age"].median())
data["CabinNum"] = data["CabinNum"].fillna(data["CabinNum"].median())

data["Deck"] = data["Deck"].replace("Unknown", np.nan)
data["Side"] = data["Side"].replace("Unknown", np.nan)

data["Deck"] = data["Deck"].fillna("Unknown")
data["Side"] = data["Side"].fillna("Unknown")

data["NoSpending"] = (data["TotalSpend"] == 0).astype(int)
data["LuxurySpend"] = data["Spa"] + data["VRDeck"] + data["FoodCourt"]
data["ServiceSpend"] = data["RoomService"] + data["ShoppingMall"]

data["FoodCourt_Ratio"] = data["FoodCourt"] / (data["TotalSpend"] + 1)
data["Spa_Ratio"] = data["Spa"] / (data["TotalSpend"] + 1)
data["VRDeck_Ratio"] = data["VRDeck"] / (data["TotalSpend"] + 1)
data["RoomService_Ratio"] = data["RoomService"] / (data["TotalSpend"] + 1)
data["ShoppingMall_Ratio"] = data["ShoppingMall"] / (data["TotalSpend"] + 1)

for col in spend_cols + ["TotalSpend", "LuxurySpend", "ServiceSpend"]:
    data[col + "_log"] = np.log1p(data[col])

# Age features
data["AgeBin"] = pd.cut(
    data["Age"],
    bins=[-1, 12, 18, 30, 45, 60, 100],
    labels=["Child", "Teen", "YoungAdult", "Adult", "MiddleAge", "Senior"]
)

data["AgeBin"] = data["AgeBin"].astype(str)
data["IsChild"] = (data["Age"] <= 12).astype(int)
data["IsTeen"] = ((data["Age"] > 12) & (data["Age"] <= 18)).astype(int)
data["IsAdult"] = (data["Age"] > 18).astype(int)


# Cabin number bins
data["CabinNumBin"] = pd.qcut(
    data["CabinNum"],
    q=5,
    labels=["CabinLow", "CabinLowMid", "CabinMid", "CabinHighMid", "CabinHigh"],
    duplicates="drop"
)

data["CabinNumBin"] = data["CabinNumBin"].astype(str)


# Interaction features
data["HomePlanet_Destination"] = (
    data["HomePlanet"].astype(str) + "_" + data["Destination"].astype(str)
)

data["Deck_Side"] = (
    data["Deck"].astype(str) + "_" + data["Side"].astype(str)
)

data["HomePlanet_Deck"] = (
    data["HomePlanet"].astype(str) + "_" + data["Deck"].astype(str)
)

data["CryoSleep_Destination"] = (
    data["CryoSleep"].astype(str) + "_" + data["Destination"].astype(str)
)

data["VIP_HomePlanet"] = (
    data["VIP"].astype(str) + "_" + data["HomePlanet"].astype(str)
)

data["CryoSleep_NoSpending"] = (
    data["CryoSleep"].astype(str) + "_" + data["NoSpending"].astype(str)
)

data["Age_TotalSpend_log"] = data["Age"] * data["TotalSpend_log"]
data["GroupSize_TotalSpend_log"] = data["GroupSize"] * data["TotalSpend_log"]
data["CabinNum_TotalSpend_log"] = data["CabinNum"] * data["TotalSpend_log"]


# KMeans cluster
cluster_features = [
    "Age",
    "RoomService_log",
    "FoodCourt_log",
    "ShoppingMall_log",
    "Spa_log",
    "VRDeck_log",
    "TotalSpend_log",
    "GroupSize",
    "CabinNum"
]

cluster_scaler = StandardScaler()
cluster_scaled = cluster_scaler.fit_transform(data[cluster_features])

kmeans = KMeans(
    n_clusters=6,
    random_state=RANDOM_STATE,
    n_init=10
)

data["SpendAgeCluster"] = kmeans.fit_predict(cluster_scaled).astype(str)


# Categorical columns
categorical_cols = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "VIP",
    "Deck",
    "Side",
    "AgeBin",
    "CabinNumBin",
    "HomePlanet_Destination",
    "Deck_Side",
    "HomePlanet_Deck",
    "CryoSleep_Destination",
    "VIP_HomePlanet",
    "CryoSleep_NoSpending",
    "SpendAgeCluster"
]

for col in categorical_cols:
    data[col] = data[col].astype(str).fillna("Unknown")


# Metadata for postprocessing
test_meta = data[data["is_train"] == 0][
    [
        "PassengerId",
        "GroupId",
        "Surname",
        "CryoSleep",
        "TotalSpend",
        "NoSpending",
        "GroupSize",
        "SurnameSize"
    ]
].copy()

test_meta.to_csv("test_postprocess_meta.csv", index=False)


# Final train/test split
drop_cols = [
    "PassengerId",
    "Cabin",
    "Name",
    "Surname"
]

data = data.drop(columns=drop_cols)


# Split train and test
train_data = data[data["is_train"] == 1].copy()
test_data = data[data["is_train"] == 0].copy()

train_data = train_data.drop(columns=["is_train"])
test_data = test_data.drop(columns=["is_train", "Transported"])

train_data["Transported"] = train_data["Transported"].astype(int)

test_data.insert(0, "PassengerId", test_passenger_id.values)


# Final dtype safety
for col in categorical_cols:
    if col in train_data.columns:
        train_data[col] = train_data[col].astype(str).fillna("Unknown")
    if col in test_data.columns:
        test_data[col] = test_data[col].astype(str).fillna("Unknown")

for col in train_data.select_dtypes(include=["bool"]).columns:
    train_data[col] = train_data[col].astype(int)

for col in test_data.select_dtypes(include=["bool"]).columns:
    test_data[col] = test_data[col].astype(int)


# Save processed files
train_data.to_csv("train_preprocessed.csv", index=False)
test_data.to_csv("test_preprocessed.csv", index=False)

with open("categorical_cols.json", "w") as f:
    json.dump(categorical_cols, f)

feature_cols = [col for col in train_data.columns if col != "Transported"]

with open("feature_cols.json", "w") as f:
    json.dump(feature_cols, f)

print("Preprocessing completed.")

Preprocessing completed.


In [3]:
import os


RANDOM_STATE = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

import time
import json
import random
import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from catboost import CatBoostClassifier

import tensorflow as tf
tf.get_logger().setLevel("ERROR")

try:
    tf.random.set_seed(RANDOM_STATE)
    tf.config.threading.set_inter_op_parallelism_threads(1)
    tf.config.threading.set_intra_op_parallelism_threads(1)
except Exception as e:
    print("TensorFlow deterministic setup warning:", e)

import tensorflow_decision_forests as tfdf

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


# Load processed data
train_data = pd.read_csv("train_preprocessed.csv")
test_data = pd.read_csv("test_preprocessed.csv")
test_meta = pd.read_csv("test_postprocess_meta.csv")

with open("categorical_cols.json", "r") as f:
    categorical_cols = json.load(f)


for col in categorical_cols:
    if col in train_data.columns:
        train_data[col] = train_data[col].astype(str).fillna("Unknown")
    if col in test_data.columns:
        test_data[col] = test_data[col].astype(str).fillna("Unknown")

for col in train_data.select_dtypes(include=["bool"]).columns:
    train_data[col] = train_data[col].astype(int)

for col in test_data.select_dtypes(include=["bool"]).columns:
    test_data[col] = test_data[col].astype(int)


# Prepare train, validation, and test data
test_passenger_id = test_data["PassengerId"].copy()

X = train_data.drop(columns=["Transported"])
y = train_data["Transported"].astype(int)

X_test = test_data.drop(columns=["PassengerId"])

full_train_df = X.copy()
full_train_df["Transported"] = y

train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=full_train_df["Transported"]
)

val_true = val_df["Transported"].astype(int).values


# Helper functions
def get_positive_probability_tfdf(pred):
    pred = np.array(pred)

    if pred.ndim == 2 and pred.shape[1] == 2:
        return pred[:, 1]

    return pred.reshape(-1)


def make_submission(filename, passenger_id, pred):
    submission = pd.DataFrame({
        "PassengerId": passenger_id,
        "Transported": pred.astype(bool)
    })

    submission.to_csv(filename, index=False)



def save_prob_submission(filename, passenger_id, prob, threshold=0.5):
    pred = (prob >= threshold).astype(bool)
    make_submission(filename, passenger_id, pred)


# Model parameters
tfdf_params = {
    "num_trees": 350,
    "max_depth": 6,
    "shrinkage": 0.045,
    "subsample": 0.85,
    "min_examples": 5
}

catboost_params = {
    "iterations": 700,
    "learning_rate": 0.025,
    "depth": 5,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.5,
    "bagging_temperature": 0.7
}


# TF-DF validation model
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_df,
    label="Transported",
    task=tfdf.keras.Task.CLASSIFICATION
)

val_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    val_df,
    label="Transported",
    task=tfdf.keras.Task.CLASSIFICATION
)

full_train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    full_train_df,
    label="Transported",
    task=tfdf.keras.Task.CLASSIFICATION
)

test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(
    X_test,
    task=tfdf.keras.Task.CLASSIFICATION
)

tfdf_model = tfdf.keras.GradientBoostedTreesModel(
    task=tfdf.keras.Task.CLASSIFICATION,
    num_trees=tfdf_params["num_trees"],
    max_depth=tfdf_params["max_depth"],
    shrinkage=tfdf_params["shrinkage"],
    subsample=tfdf_params["subsample"],
    min_examples=tfdf_params["min_examples"],
    random_seed=RANDOM_STATE,
    verbose=0
)

tfdf_model.compile(metrics=["accuracy"])

start_tfdf_train = time.time()
tfdf_model.fit(train_ds, verbose=0)
tfdf_training_time = time.time() - start_tfdf_train

start_tfdf_val_inf = time.time()
tfdf_val_prob_raw = tfdf_model.predict(val_ds, verbose=0)
tfdf_val_inference_time = time.time() - start_tfdf_val_inf

tfdf_val_prob = get_positive_probability_tfdf(tfdf_val_prob_raw)
tfdf_val_pred = (tfdf_val_prob >= 0.5).astype(int)
tfdf_val_acc = accuracy_score(val_true, tfdf_val_pred)


# CatBoost validation model
cat_train_all = train_data.copy()
cat_test_all = test_data.copy()

for col in categorical_cols:
    if col in cat_train_all.columns:
        cat_train_all[col] = cat_train_all[col].astype(str).fillna("Unknown")
    if col in cat_test_all.columns:
        cat_test_all[col] = cat_test_all[col].astype(str).fillna("Unknown")

cat_X = cat_train_all.drop(columns=["Transported"])
cat_y = cat_train_all["Transported"].astype(int)
cat_X_test = cat_test_all.drop(columns=["PassengerId"])

cat_X_train, cat_X_val, cat_y_train, cat_y_val = train_test_split(
    cat_X,
    cat_y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=cat_y
)

cat_features = [col for col in categorical_cols if col in cat_X.columns]

cat_model = CatBoostClassifier(
    iterations=catboost_params["iterations"],
    learning_rate=catboost_params["learning_rate"],
    depth=catboost_params["depth"],
    l2_leaf_reg=catboost_params["l2_leaf_reg"],
    random_strength=catboost_params["random_strength"],
    bagging_temperature=catboost_params["bagging_temperature"],
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=RANDOM_STATE,
    thread_count=1,
    verbose=False,
    allow_writing_files=False
)

start_cat_train = time.time()
cat_model.fit(
    cat_X_train,
    cat_y_train,
    cat_features=cat_features
)
cat_training_time = time.time() - start_cat_train

start_cat_val_inf = time.time()
cat_val_prob = cat_model.predict_proba(cat_X_val)[:, 1]
cat_val_inference_time = time.time() - start_cat_val_inf

cat_val_pred = (cat_val_prob >= 0.5).astype(int)
cat_val_acc = accuracy_score(cat_y_val, cat_val_pred)


# Validation soft voting
base_val_prob = 0.93 * tfdf_val_prob + 0.07 * cat_val_prob
base_val_pred = (base_val_prob >= 0.5).astype(int)
base_val_acc = accuracy_score(val_true, base_val_pred)


# Final TF-DF model
final_tfdf_model = tfdf.keras.GradientBoostedTreesModel(
    task=tfdf.keras.Task.CLASSIFICATION,
    num_trees=tfdf_params["num_trees"],
    max_depth=tfdf_params["max_depth"],
    shrinkage=tfdf_params["shrinkage"],
    subsample=tfdf_params["subsample"],
    min_examples=tfdf_params["min_examples"],
    random_seed=RANDOM_STATE,
    verbose=0
)

final_tfdf_model.compile(metrics=["accuracy"])

start_final_tfdf = time.time()
final_tfdf_model.fit(full_train_ds, verbose=0)
final_tfdf_training_time = time.time() - start_final_tfdf

start_tfdf_test = time.time()
tfdf_test_prob_raw = final_tfdf_model.predict(test_ds, verbose=0)
tfdf_test_inference_time = time.time() - start_tfdf_test

tfdf_test_prob = get_positive_probability_tfdf(tfdf_test_prob_raw)

# Final CatBoost model
final_cat_model = CatBoostClassifier(
    iterations=catboost_params["iterations"],
    learning_rate=catboost_params["learning_rate"],
    depth=catboost_params["depth"],
    l2_leaf_reg=catboost_params["l2_leaf_reg"],
    random_strength=catboost_params["random_strength"],
    bagging_temperature=catboost_params["bagging_temperature"],
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=RANDOM_STATE,
    thread_count=1,
    verbose=False,
    allow_writing_files=False
)

start_final_cat = time.time()
final_cat_model.fit(
    cat_X,
    cat_y,
    cat_features=cat_features
)
final_cat_training_time = time.time() - start_final_cat

start_cat_test = time.time()
cat_test_prob = final_cat_model.predict_proba(cat_X_test)[:, 1]
cat_test_inference_time = time.time() - start_cat_test


# Save models and configuration
final_tfdf_model.save("tfdf_model")
final_cat_model.save_model("catboost_b_model.cbm")

with open("model_config.json", "w") as f:
    json.dump(
        {
            "random_state": RANDOM_STATE,
            "tfdf_params": tfdf_params,
            "catboost_params": catboost_params,
            "base_weight": {
                "tfdf": 0.93,
                "catboost_b": 0.07
            },
            "reproducibility": {
                "PYTHONHASHSEED": str(RANDOM_STATE),
                "TF_DETERMINISTIC_OPS": "1",
                "OMP_NUM_THREADS": "1",
                "TF_NUM_INTRAOP_THREADS": "1",
                "TF_NUM_INTEROP_THREADS": "1",
                "catboost_thread_count": 1
            }
        },
        f,
        indent=4
    )


# Test probability fusion
base_test_prob = 0.93 * tfdf_test_prob + 0.07 * cat_test_prob
base_pred = (base_test_prob >= 0.5).astype(bool)

save_prob_submission(
    "best_base_model_prediction_93_07.csv",
    test_passenger_id,
    base_test_prob,
    threshold=0.5
)


# Rule-based postprocessing
post_df = test_meta.copy()
post_df["base_prob"] = base_test_prob
post_df["base_pred"] = base_pred.astype(bool)

post_df["CryoSleep"] = post_df["CryoSleep"].astype(str)
post_df["Surname"] = post_df["Surname"].astype(str)

post_df["high_true"] = post_df["base_prob"] >= 0.55
post_df["high_false"] = post_df["base_prob"] <= 0.45

group_summary = post_df.groupby("GroupId").agg(
    group_size=("PassengerId", "count"),
    high_true_count=("high_true", "sum"),
    high_false_count=("high_false", "sum"),
    mean_prob=("base_prob", "mean")
).reset_index()

surname_summary = post_df.groupby("Surname").agg(
    surname_size=("PassengerId", "count"),
    surname_high_true_count=("high_true", "sum"),
    surname_high_false_count=("high_false", "sum"),
    surname_mean_prob=("base_prob", "mean")
).reset_index()

post_df = post_df.merge(group_summary, on="GroupId", how="left")
post_df = post_df.merge(surname_summary, on="Surname", how="left")


def apply_light_rules(df):
    adjusted = df["base_pred"].copy()
    changed_reason = ["none"] * len(df)

    for i, row in df.iterrows():
        prob = row["base_prob"]

        if not (0.49 <= prob <= 0.51):
            continue

        cryo_true = row["CryoSleep"] == "True"
        no_spending = row["TotalSpend"] == 0

        if cryo_true and no_spending and prob >= 0.495:
            adjusted.iloc[i] = True
            changed_reason[i] = "light_cryo_no_spend_true"
            continue

        if (not cryo_true) and row["TotalSpend"] > 0 and prob <= 0.505:
            adjusted.iloc[i] = False
            changed_reason[i] = "light_awake_spend_false"
            continue

        if row["group_size"] >= 2:
            if row["high_true_count"] >= 2 and row["high_false_count"] == 0:
                adjusted.iloc[i] = True
                changed_reason[i] = "light_group_true"
                continue

            if row["high_false_count"] >= 2 and row["high_true_count"] == 0:
                adjusted.iloc[i] = False
                changed_reason[i] = "light_group_false"
                continue

    return adjusted.astype(bool), changed_reason


def apply_medium_rules(df):
    adjusted = df["base_pred"].copy()
    changed_reason = ["none"] * len(df)

    for i, row in df.iterrows():
        prob = row["base_prob"]

        if not (0.48 <= prob <= 0.52):
            continue

        cryo_true = row["CryoSleep"] == "True"
        no_spending = row["TotalSpend"] == 0

        if cryo_true and no_spending and prob >= 0.485:
            adjusted.iloc[i] = True
            changed_reason[i] = "medium_cryo_no_spend_true"
            continue

        if (not cryo_true) and row["TotalSpend"] > 0 and prob <= 0.515:
            adjusted.iloc[i] = False
            changed_reason[i] = "medium_awake_spend_false"
            continue

        if row["group_size"] >= 2:
            if row["mean_prob"] >= 0.56 and row["high_true_count"] >= 1:
                adjusted.iloc[i] = True
                changed_reason[i] = "medium_group_true"
                continue

            if row["mean_prob"] <= 0.44 and row["high_false_count"] >= 1:
                adjusted.iloc[i] = False
                changed_reason[i] = "medium_group_false"
                continue

    return adjusted.astype(bool), changed_reason


def apply_group_family_rules(df):
    adjusted = df["base_pred"].copy()
    changed_reason = ["none"] * len(df)

    for i, row in df.iterrows():
        prob = row["base_prob"]

        if not (0.48 <= prob <= 0.52):
            continue

        if row["group_size"] >= 2:
            if row["high_true_count"] >= 2 and row["mean_prob"] >= 0.55:
                adjusted.iloc[i] = True
                changed_reason[i] = "group_family_group_true"
                continue

            if row["high_false_count"] >= 2 and row["mean_prob"] <= 0.45:
                adjusted.iloc[i] = False
                changed_reason[i] = "group_family_group_false"
                continue

        if row["surname_size"] >= 3 and row["Surname"] != "Unknown":
            if row["surname_high_true_count"] >= 3 and row["surname_mean_prob"] >= 0.56:
                adjusted.iloc[i] = True
                changed_reason[i] = "group_family_surname_true"
                continue

            if row["surname_high_false_count"] >= 3 and row["surname_mean_prob"] <= 0.44:
                adjusted.iloc[i] = False
                changed_reason[i] = "group_family_surname_false"
                continue

    return adjusted.astype(bool), changed_reason


# Generate postprocessed submissions
light_pred, light_reason = apply_light_rules(post_df)
medium_pred, medium_reason = apply_medium_rules(post_df)
group_family_pred, group_family_reason = apply_group_family_rules(post_df)

make_submission(
    "postprocess_rule_light_prediction.csv",
    test_passenger_id,
    light_pred
)


# Save probability file
prob_output = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "TFDF_Probability": tfdf_test_prob,
    "CatBoost_b_Probability": cat_test_prob,
    "Base_93_07_Probability": base_test_prob
})

prob_output.to_csv("postprocess_probabilities.csv", index=False)

print("Training completed. Necessary files saved: postprocess_rule_light_prediction.csv and postprocess_probabilities.csv.")
print("Base validation accuracy:", round(base_val_acc, 5))

[WARNING 26-05-18 11:52:28.0784 CST gradient_boosted_trees.cc:1818] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-05-18 11:52:28.0789 CST gradient_boosted_trees.cc:1829] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-05-18 11:52:28.0789 CST gradient_boosted_trees.cc:1843] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
[INFO 26-05-18 11:52:31.0317 CST kernel.cc:1243] Loading model from path /var/folders/xc/lp8d521n2px7lx0mjg8cpfdr0000gn/T/tmpmghz84z2/model/ with prefix d4a093ca91d94949
[INFO 26-05-18 11:52:31.0381 CST abstract_model.cc:1312] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 26-05-18 11:52:31.0381 CST kernel.cc:1075] Use fast generic engine


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


[WARNING 26-05-18 11:52:39.9245 CST gradient_boosted_trees.cc:1818] "goss_alpha" set but "sampling_method" not equal to "GOSS".
[WARNING 26-05-18 11:52:39.9246 CST gradient_boosted_trees.cc:1829] "goss_beta" set but "sampling_method" not equal to "GOSS".
[WARNING 26-05-18 11:52:39.9246 CST gradient_boosted_trees.cc:1843] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
[INFO 26-05-18 11:52:41.7636 CST kernel.cc:1243] Loading model from path /var/folders/xc/lp8d521n2px7lx0mjg8cpfdr0000gn/T/tmp2we3w9k8/model/ with prefix 64224f272fdf4f8c
[INFO 26-05-18 11:52:41.7715 CST abstract_model.cc:1312] Engine "GradientBoostedTreesQuickScorerExtended" built
[INFO 26-05-18 11:52:41.7715 CST kernel.cc:1075] Use fast generic engine


Training completed. Necessary files saved: postprocess_rule_light_prediction.csv and postprocess_probabilities.csv.
Base validation accuracy: 0.82116


In [4]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd


# Input files
BEST_PREDICTION_PATH = "postprocess_rule_light_prediction.csv"
PROB_PATH = "postprocess_probabilities.csv"
META_PATH = "test_postprocess_meta.csv"
TEST_PATH = "test.csv"


# V10 threshold setting
VERSION = "v10"
LOW = 0.4725
HIGH = 0.5275
TAG = "04725_05275"


# Load files
best_sub = pd.read_csv(BEST_PREDICTION_PATH)
prob_df = pd.read_csv(PROB_PATH)
meta_df = pd.read_csv(META_PATH)
test_raw = pd.read_csv(TEST_PATH)


# Merge base prediction, probabilities, and metadata
df = best_sub.copy()
df = df.merge(prob_df, on="PassengerId", how="left")
df = df.merge(meta_df, on="PassengerId", how="left")

if df["Base_93_07_Probability"].isna().sum() > 0:
    raise ValueError("Some PassengerId values did not match postprocess_probabilities.csv.")

if df["GroupId"].isna().sum() > 0:
    raise ValueError("Some PassengerId values did not match test_postprocess_meta.csv.")

df["base_postprocess_pred"] = df["Transported"].astype(bool)
df["base_prob"] = df["Base_93_07_Probability"].astype(float)

df["GroupId"] = df["GroupId"].astype(str)
df["Surname"] = df["Surname"].astype(str)
df["CryoSleep"] = df["CryoSleep"].astype(str)


# Add raw test information
raw_extra = test_raw.copy()

raw_extra["Cabin"] = raw_extra["Cabin"].fillna("Unknown/Unknown/Unknown")
raw_extra["Deck"] = raw_extra["Cabin"].astype(str).str.split("/").str[0]
raw_extra["CabinNum"] = raw_extra["Cabin"].astype(str).str.split("/").str[1]
raw_extra["Side"] = raw_extra["Cabin"].astype(str).str.split("/").str[2]

raw_extra["CabinNum"] = pd.to_numeric(raw_extra["CabinNum"], errors="coerce")
raw_extra["Deck"] = raw_extra["Deck"].fillna("Unknown")
raw_extra["Side"] = raw_extra["Side"].fillna("Unknown")

raw_extra["HomePlanet"] = raw_extra["HomePlanet"].fillna("Unknown")
raw_extra["Destination"] = raw_extra["Destination"].fillna("Unknown")

spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

for col in spend_cols:
    raw_extra[col] = raw_extra[col].fillna(0)

raw_extra["RawTotalSpend"] = raw_extra[spend_cols].sum(axis=1)

raw_extra["SpendCount"] = (
    (raw_extra["RoomService"] > 0).astype(int)
    + (raw_extra["FoodCourt"] > 0).astype(int)
    + (raw_extra["ShoppingMall"] > 0).astype(int)
    + (raw_extra["Spa"] > 0).astype(int)
    + (raw_extra["VRDeck"] > 0).astype(int)
)

df = df.merge(
    raw_extra[
        [
            "PassengerId",
            "Deck",
            "CabinNum",
            "Side",
            "HomePlanet",
            "Destination",
            "RoomService",
            "FoodCourt",
            "ShoppingMall",
            "Spa",
            "VRDeck",
            "RawTotalSpend",
            "SpendCount"
        ]
    ],
    on="PassengerId",
    how="left"
)


# Group-level and surname-level summaries
df["high_true"] = df["base_prob"] >= 0.55
df["high_false"] = df["base_prob"] <= 0.45

group_summary = df.groupby("GroupId").agg(
    group_size=("PassengerId", "count"),
    group_sum_prob=("base_prob", "sum"),
    group_mean_prob=("base_prob", "mean"),
    group_max_prob=("base_prob", "max"),
    group_min_prob=("base_prob", "min"),
    group_std_prob=("base_prob", "std"),
    group_high_true_count=("high_true", "sum"),
    group_high_false_count=("high_false", "sum")
).reset_index()

group_summary["group_std_prob"] = group_summary["group_std_prob"].fillna(0)

df = df.merge(group_summary, on="GroupId", how="left")

df["group_other_mean_prob"] = np.where(
    df["group_size"] > 1,
    (df["group_sum_prob"] - df["base_prob"]) / (df["group_size"] - 1),
    df["base_prob"]
)

df["group_other_mean_prob"] = df["group_other_mean_prob"].fillna(df["base_prob"])


# Surname-level probability summary
surname_summary = df.groupby("Surname").agg(
    surname_size=("PassengerId", "count"),
    surname_sum_prob=("base_prob", "sum"),
    surname_mean_prob=("base_prob", "mean"),
    surname_max_prob=("base_prob", "max"),
    surname_min_prob=("base_prob", "min"),
    surname_std_prob=("base_prob", "std"),
    surname_high_true_count=("high_true", "sum"),
    surname_high_false_count=("high_false", "sum")
).reset_index()

surname_summary["surname_std_prob"] = surname_summary["surname_std_prob"].fillna(0)

df = df.merge(surname_summary, on="Surname", how="left")

df["surname_other_mean_prob"] = np.where(
    df["surname_size"] > 1,
    (df["surname_sum_prob"] - df["base_prob"]) / (df["surname_size"] - 1),
    df["base_prob"]
)

df["surname_other_mean_prob"] = df["surname_other_mean_prob"].fillna(df["base_prob"])


# Candidate builder
def build_threshold_candidates(data, low, high):
    boundary_df = data[
        (data["base_prob"] >= low)
        & (data["base_prob"] <= high)
    ].copy()

    candidates = []

    for i, row in boundary_df.iterrows():
        prob = float(row["base_prob"])
        current_pred = bool(row["base_postprocess_pred"])
        margin_bonus = ((high - low) / 2) - abs(prob - 0.5)

        # Rule 1: group consistency
        if row["group_size"] >= 2:
            other_mean = float(row["group_other_mean_prob"])

            if (not current_pred) and other_mean >= 0.62:
                strength = (
                    (other_mean - 0.5)
                    + margin_bonus
                    + 0.01 * row["group_high_true_count"]
                )

                candidates.append({
                    "index": i,
                    "PassengerId": row["PassengerId"],
                    "old_pred": current_pred,
                    "new_pred": True,
                    "candidate_type": "group_false_to_true",
                    "candidate_group": "group",
                    "base_prob": prob,
                    "group_other_mean_prob": other_mean,
                    "strength": strength
                })

            if current_pred and other_mean <= 0.38:
                strength = (
                    (0.5 - other_mean)
                    + margin_bonus
                    + 0.01 * row["group_high_false_count"]
                )

                candidates.append({
                    "index": i,
                    "PassengerId": row["PassengerId"],
                    "old_pred": current_pred,
                    "new_pred": False,
                    "candidate_type": "group_true_to_false",
                    "candidate_group": "group",
                    "base_prob": prob,
                    "group_other_mean_prob": other_mean,
                    "strength": strength
                })

        # Rule 2: CryoSleep and spending
        cryo_true = row["CryoSleep"] == "True"
        total_spend = float(row["TotalSpend"])

        if (not current_pred) and cryo_true and total_spend == 0:
            strength = (
                margin_bonus
                + 0.02
                + max(0, row["group_other_mean_prob"] - 0.5)
            )

            candidates.append({
                "index": i,
                "PassengerId": row["PassengerId"],
                "old_pred": current_pred,
                "new_pred": True,
                "candidate_type": "cryo_no_spend_false_to_true",
                "candidate_group": "cryo_spend",
                "base_prob": prob,
                "CryoSleep": row["CryoSleep"],
                "TotalSpend": total_spend,
                "strength": strength
            })

        if current_pred and (not cryo_true) and total_spend > 0:
            strength = (
                margin_bonus
                + 0.02
                + max(0, 0.5 - row["group_other_mean_prob"])
            )

            candidates.append({
                "index": i,
                "PassengerId": row["PassengerId"],
                "old_pred": current_pred,
                "new_pred": False,
                "candidate_type": "awake_spend_true_to_false",
                "candidate_group": "cryo_spend",
                "base_prob": prob,
                "CryoSleep": row["CryoSleep"],
                "TotalSpend": total_spend,
                "strength": strength
            })

        # Rule 3: model disagreement
        tfdf_prob = float(row["TFDF_Probability"])
        cat_prob = float(row["CatBoost_b_Probability"])

        if current_pred and cat_prob <= 0.42 and tfdf_prob >= 0.50:
            strength = (
                (0.42 - cat_prob)
                + abs(tfdf_prob - cat_prob)
                + margin_bonus
            )

            candidates.append({
                "index": i,
                "PassengerId": row["PassengerId"],
                "old_pred": current_pred,
                "new_pred": False,
                "candidate_type": "cat_disagrees_true_to_false",
                "candidate_group": "model_disagreement_negative",
                "base_prob": prob,
                "TFDF_Probability": tfdf_prob,
                "CatBoost_b_Probability": cat_prob,
                "model_abs_diff": abs(tfdf_prob - cat_prob),
                "strength": strength
            })

        # Rule 4: surname/family consistency
        surname_other_mean = float(row["surname_other_mean_prob"])

        if (
            current_pred
            and row["Surname"] != "Unknown"
            and row["surname_size"] >= 3
            and surname_other_mean <= 0.42
        ):
            strength = (
                (0.5 - surname_other_mean)
                + margin_bonus
                + 0.01 * row["surname_high_false_count"]
            )

            candidates.append({
                "index": i,
                "PassengerId": row["PassengerId"],
                "old_pred": current_pred,
                "new_pred": False,
                "candidate_type": "surname_true_to_false",
                "candidate_group": "surname_negative",
                "base_prob": prob,
                "Surname": row["Surname"],
                "surname_other_mean_prob": surname_other_mean,
                "surname_size": row["surname_size"],
                "strength": strength
            })

    raw_candidates = pd.DataFrame(candidates)

    if len(raw_candidates) > 0:
        raw_candidates = raw_candidates.sort_values(
            by="strength",
            ascending=False
        ).reset_index(drop=True)

        unique_candidates = raw_candidates.drop_duplicates(
            subset=["index"],
            keep="first"
        ).reset_index(drop=True)

        unique_candidates = unique_candidates.sort_values(
            by="strength",
            ascending=False
        ).reset_index(drop=True)
    else:
        unique_candidates = pd.DataFrame()

    return boundary_df, raw_candidates, unique_candidates


# Generate v10 candidates only
boundary_df, raw_candidates, unique_candidates = build_threshold_candidates(
    df,
    LOW,
    HIGH
)

print(f"Candidate generation completed. Unique candidates: {len(unique_candidates)}.")

Candidate generation completed. Unique candidates: 88.


In [5]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

# Load base prediction
BEST_BASE_PATH = "postprocess_rule_light_prediction.csv"

best_base = pd.read_csv(BEST_BASE_PATH)

if "PassengerId" not in best_base.columns or "Transported" not in best_base.columns:
    raise ValueError("Base model submission must contain PassengerId and Transported columns.")

best_base["Transported"] = best_base["Transported"].astype(bool)
base_pred = best_base["Transported"].copy()


# Final threshold version
VERSION = "v10"
LOW = 0.4725
HIGH = 0.5275
TAG = "04725_05275"


# Apply selected candidates
if "unique_candidates" not in globals():
    raise RuntimeError("unique_candidates not found. Please run Cell 3 first.")

candidates = unique_candidates.copy()

pred = base_pred.copy()

if len(candidates) > 0:
    candidates["index"] = candidates["index"].astype(int)
    candidates["new_pred"] = candidates["new_pred"].astype(bool)

    candidates = candidates.sort_values(
        by="strength",
        ascending=False
    ).reset_index(drop=True)

    candidates = candidates.drop_duplicates(
        subset=["index"],
        keep="first"
    ).reset_index(drop=True)

    for _, row in candidates.iterrows():
        pred.iloc[int(row["index"])] = bool(row["new_pred"])


# Save final submission
submission_file = (
    f"submission_{VERSION}_"
    f"flip_all_candidates_threshold_{TAG}.csv"
)

submission = pd.DataFrame({
    "PassengerId": best_base["PassengerId"],
    "Transported": pred.astype(bool)
})

submission.to_csv(submission_file, index=False)

changed_count = int((pred != base_pred).sum())


print(f"Final submission saved: {submission_file}")
print(f"Changed from base prediction: {changed_count}")

Final submission saved: submission_v10_flip_all_candidates_threshold_04725_05275.csv
Changed from base prediction: 88
